# Part 2 — RFM Segmentation & Retention Strategy
This notebook is generated from `build_part2.py` and captures the segment evidence from the actual dataset run.

## Data Loading

The workflow loads the raw customer, order, support, web snapshot, churn label, and intervention files from relative paths.

| dataset                  |   rows |   columns |
|:-------------------------|-------:|----------:|
| customers.csv            |   2400 |         9 |
| orders.csv               |  10009 |        10 |
| support_tickets.csv      |   1921 |         8 |
| web_events_snapshot.csv  |   2400 |        10 |
| churn_labels.csv         |   2400 |         4 |
| intervention_history.csv |   2400 |         5 |

In [ ]:
from pathlib import Path
import pandas as pd

SNAPSHOT_DATE = pd.Timestamp('2025-09-30')
candidates = [
    Path('data'),
    Path('../d2c churn data package/d2c churn data package'),
]
data_dir = next(path for path in candidates if (path / 'customers.csv').exists())
customers = pd.read_csv(data_dir / 'customers.csv', parse_dates=['signup_date'])
orders = pd.read_csv(data_dir / 'orders.csv', parse_dates=['order_date'])
support = pd.read_csv(data_dir / 'support_tickets.csv', parse_dates=['ticket_date'])
web = pd.read_csv(data_dir / 'web_events_snapshot.csv', parse_dates=['snapshot_date'])
interventions = pd.read_csv(data_dir / 'intervention_history.csv', parse_dates=['snapshot_date'])
customers.shape, orders.shape, support.shape, web.shape, interventions.shape

## RFM Feature Creation

The segmentation starts with recency, frequency, and monetary features built from pre-snapshot orders. The sample below shows the actual engineered RFM fields used in the run.

| customer_id   |   recency_days |   frequency_180d |   monetary_180d |   r_score |   f_score |   m_score |   rfm_total |
|:--------------|---------------:|-----------------:|----------------:|----------:|----------:|----------:|------------:|
| CUST00001     |            107 |                1 |          362.73 |         2 |         1 |         1 |           4 |
| CUST00002     |             40 |                1 |          581    |         4 |         1 |         2 |           7 |
| CUST00003     |            171 |                1 |          649.98 |         1 |         1 |         2 |           4 |
| CUST00004     |            131 |                1 |         1604.04 |         2 |         1 |         4 |           7 |
| CUST00005     |             38 |                3 |         1781.9  |         4 |         5 |         4 |          13 |
| CUST00006     |             51 |                4 |         2989.58 |         3 |         5 |         5 |          13 |
| CUST00007     |              3 |                1 |          719.33 |         5 |         1 |         3 |           9 |
| CUST00008     |             47 |                3 |         2449.11 |         3 |         5 |         5 |          13 |
| CUST00009     |             31 |                1 |          376.85 |         4 |         1 |         2 |           7 |
| CUST00010     |              9 |                1 |          636.8  |         5 |         1 |         2 |           8 |
| CUST00011     |              1 |                1 |          508.08 |         5 |         1 |         2 |           8 |
| CUST00012     |             50 |                1 |          978.88 |         3 |         1 |         3 |           7 |

In [ ]:
pre_orders = orders.loc[orders['order_date'] <= SNAPSHOT_DATE].copy()
orders_180d = pre_orders.loc[pre_orders['order_date'] >= SNAPSHOT_DATE - pd.Timedelta(days=180)].copy()

last_order = pre_orders.groupby('customer_id')['order_date'].max().rename('last_order_date')
rfm = customers[['customer_id', 'signup_date']].merge(last_order, on='customer_id', how='left')
rfm['recency_days'] = (SNAPSHOT_DATE - rfm['last_order_date']).dt.days
rfm['recency_days'] = rfm['recency_days'].fillna((SNAPSHOT_DATE - rfm['signup_date']).dt.days + 999)
rfm = rfm.merge(
    orders_180d.groupby('customer_id').agg(
        frequency_180d=('order_id', 'nunique'),
        monetary_180d=('gross_amount', 'sum'),
    ).reset_index(),
    on='customer_id',
    how='left',
)
rfm.head()

## Additional Behavioural / Support Signals

RFM is combined with non-RFM evidence from support, returns, campaign response, and digital activity.

| segment_name       |   avg_sessions_30d |   avg_ticket_count_90d |   avg_return_rate_180d |   avg_discount_pct_180d |   avg_campaign_clicks_30d |
|:-------------------|-------------------:|-----------------------:|-----------------------:|------------------------:|--------------------------:|
| Dormant At-Risk    |            1.01171 |             0.00234192 |             0.0163934  |                0.150317 |                  0.381733 |
| Discount-Sensitive |            6.47321 |             0.25       |             0.077381   |                0.407016 |                  1.64286  |
| Service Recovery   |            5.28    |             0.88       |             0.31625    |                0.271158 |                  0.395    |
| Mixed Watchlist    |            5.54877 |             0.039953   |             0.00705053 |                0.233859 |                  0.548766 |
| Growth Potential   |            8.29412 |             0.266968   |             0.0859729  |                0.284646 |                  1.1086   |
| Loyal Core         |            7.15862 |             0.568966   |             0.150718   |                0.27761  |                  0.641379 |
| Champions          |            7.55853 |             0.29097    |             0          |                0.273617 |                  0.785953 |

## Segmentation Logic

The table below shows the exact ordered rules used to assign segments.

| segment_name       | exact_rule                                                                                                                                                                                 | signals_used                                              |
|:-------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------|
| Champions          | Assigned if r_score >= 4, f_score >= 4, m_score >= 4, return_rate_180d <= 0.10, and ticket_count_90d <= 1.                                                                                 | RFM + returns + support complaints                        |
| Loyal Core         | Assigned if not already Champions and r_score >= 4, f_score >= 3, and m_score >= 3.                                                                                                        | RFM                                                       |
| Growth Potential   | Assigned if not matched above, r_score >= 4, high_engagement is true, and either f_score <= 2 or m_score <= 2. `high_engagement` means sessions_30d >= 8 or campaign_clicks_30d >= 1.      | RFM + app/web activity + campaign engagement              |
| Discount-Sensitive | Assigned if not matched above, avg_discount_pct_180d >= 0.34, campaign_clicks_30d > 0, and r_score >= 2.                                                                                   | Discount usage + campaign engagement + recency            |
| Service Recovery   | Assigned if not matched above, r_score >= 2, and service_friction is true. service_friction means (ticket_count_90d >= 1 and negative_ticket_rate_90d >= 0.50) or return_rate_180d > 0.25. | Support complaints + ticket sentiment + returns + recency |
| Dormant At-Risk    | Assigned if not matched above, r_score <= 2, and sessions_30d <= 2.                                                                                                                        | Recency + app/web activity                                |
| Mixed Watchlist    | Catch-all segment for customers not captured by any higher-priority rule above.                                                                                                            | Residual mixed signal set                                 |

In [ ]:
def assign_segment(row):
    if row.r_score >= 4 and row.f_score >= 4 and row.m_score >= 4 and row.return_rate_180d <= 0.10 and row.ticket_count_90d <= 1:
        return 'Champions'
    if row.r_score >= 4 and row.f_score >= 3 and row.m_score >= 3:
        return 'Loyal Core'
    if row.r_score >= 4 and row.high_engagement and (row.f_score <= 2 or row.m_score <= 2):
        return 'Growth Potential'
    if row.discount_sensitive_flag and row.r_score >= 2:
        return 'Discount-Sensitive'
    if row.r_score >= 2 and row.service_friction:
        return 'Service Recovery'
    if row.r_score <= 2 and row.sessions_30d <= 2:
        return 'Dormant At-Risk'
    return 'Mixed Watchlist'


## Segment Summary

| segment_name       |   customers |   observed_churn_pct |   avg_recency_days |   avg_frequency_180d |   avg_monetary_180d |
|:-------------------|------------:|---------------------:|-------------------:|---------------------:|--------------------:|
| Dormant At-Risk    |         427 |                 86.4 |           189.817  |             0.779859 |             597.812 |
| Discount-Sensitive |         112 |                 60.7 |            85.3839 |             1.60714  |            1068.51  |
| Service Recovery   |         200 |                 56.5 |            69.48   |             1.81     |            1421.51  |
| Mixed Watchlist    |         851 |                 53.9 |           104.145  |             1.34195  |             991.309 |
| Growth Potential   |         221 |                 21.7 |            20.0136 |             1.0362   |             685.719 |
| Loyal Core         |         290 |                 15.2 |            21.4172 |             2.32414  |            1611.75  |
| Champions          |         299 |                  8.7 |            19.8328 |             3.04348  |            2413.32  |

## Visual Evidence

### Customer Count by Segment
![Customer Count by Segment](charts/01_segment_counts.png)

The segmentation covers the full 2,400-customer base and keeps the largest ambiguous pool in a separate watchlist instead of forcing a false-precision label.

### Observed 60-Day Churn Rate by Segment
![Observed 60-Day Churn Rate by Segment](charts/02_segment_churn_rates.png)

The segment definitions are not arbitrary: churn separates clearly between Champions/Loyal Core and the risk-heavy segments.

### Segment Feature Profile: Recency, Frequency, and Monetary
![Segment Feature Profile: Recency, Frequency, and Monetary](charts/03_segment_rfm_heatmap.png)

The segments are anchored in RFM behavior first, then sharpened with support and engagement signals.

### Segment Value at Risk: Average Monetary vs Observed Churn
![Segment Value at Risk: Average Monetary vs Observed Churn](charts/04_segment_value_at_risk.png)

This chart helps separate cold-but-low-value groups from segments where the brand still has material spend worth protecting.

## Final Segment Assignment Sample

The full customer-level output is written to `segments.csv`. The sample below shows actual final assignments with the key features used in the segmentation.

| customer_id   | segment_name       |   recency_days |   frequency_180d |   monetary_180d |   sessions_30d |   campaign_clicks_30d |   ticket_count_90d |   return_rate_180d |
|:--------------|:-------------------|---------------:|-----------------:|----------------:|---------------:|----------------------:|-------------------:|-------------------:|
| CUST00005     | Champions          |             38 |                3 |         1781.9  |             18 |                     1 |                  0 |               0    |
| CUST00029     | Champions          |             18 |                3 |         1382.04 |              1 |                     0 |                  0 |               0    |
| CUST00018     | Discount-Sensitive |            111 |                1 |          329.21 |              3 |                     2 |                  0 |               0    |
| CUST00041     | Discount-Sensitive |             54 |                1 |          366.99 |              5 |                     2 |                  0 |               0    |
| CUST00001     | Dormant At-Risk    |            107 |                1 |          362.73 |              1 |                     0 |                  0 |               0    |
| CUST00003     | Dormant At-Risk    |            171 |                1 |          649.98 |              1 |                     0 |                  0 |               0    |
| CUST00002     | Growth Potential   |             40 |                1 |          581    |              8 |                     0 |                  1 |               0    |
| CUST00007     | Growth Potential   |              3 |                1 |          719.33 |             11 |                     3 |                  0 |               0    |
| CUST00015     | Loyal Core         |             35 |                2 |         1523.14 |              6 |                     0 |                  0 |               0    |
| CUST00019     | Loyal Core         |              9 |                2 |         1598.93 |              5 |                     1 |                  0 |               0    |
| CUST00008     | Mixed Watchlist    |             47 |                3 |         2449.11 |              2 |                     0 |                  0 |               0    |
| CUST00012     | Mixed Watchlist    |             50 |                1 |          978.88 |              3 |                     0 |                  0 |               0    |
| CUST00006     | Service Recovery   |             51 |                4 |         2989.58 |              2 |                     0 |                  2 |               0.25 |
| CUST00021     | Service Recovery   |             77 |                2 |         1059.3  |              3 |                     0 |                  1 |               0    |

# Retention Strategy

## Segment Logic

The segmentation uses classic RFM signals first and then sharpens them with support friction, return behavior, and digital engagement:

1. `Champions`: recent, frequent, high-spend buyers with low friction.
2. `Loyal Core`: strong recent value but not quite at the champion bar.
3. `Growth Potential`: fresh customers with strong engagement but lighter spend depth.
4. `Discount-Sensitive`: customers whose recent shopping pattern leans on discounts and campaign response.
5. `Service Recovery`: customers showing service pain through tickets, negative sentiment, or returns.
6. `Dormant At-Risk`: stale recency plus very low current activity.
7. `Mixed Watchlist`: remaining customers where the signal is real but not clean enough for a heavier intervention.

## RFM Feature Construction

1. `recency_days`: days between the snapshot date (`2025-09-30`) and the customer’s latest pre-snapshot order.
2. `frequency_180d`: count of distinct pre-snapshot orders in the 180 days before the snapshot.
3. `monetary_180d`: total pre-snapshot gross spend in the 180 days before the snapshot.
4. `r_score`, `f_score`, `m_score`: quintile scores from 1 to 5, where higher is better for recency freshness, order frequency, and spend.

## Data-Driven Thresholds Used In Segmentation

| threshold_name                             |   value | used_for                                                         |
|:-------------------------------------------|--------:|:-----------------------------------------------------------------|
| High-engagement sessions cutoff            |    8    | `sessions_30d >= cutoff` in the Growth Potential rule            |
| High-engagement campaign-click cutoff      |    1    | `campaign_clicks_30d >= cutoff` in the Growth Potential rule     |
| Discount-sensitive average-discount cutoff |    0.34 | `avg_discount_pct_180d >= cutoff` in the Discount-Sensitive rule |

## Exact Segment Rules

| segment_name       | exact_rule                                                                                                                                                                                 | signals_used                                              |
|:-------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------|
| Champions          | Assigned if r_score >= 4, f_score >= 4, m_score >= 4, return_rate_180d <= 0.10, and ticket_count_90d <= 1.                                                                                 | RFM + returns + support complaints                        |
| Loyal Core         | Assigned if not already Champions and r_score >= 4, f_score >= 3, and m_score >= 3.                                                                                                        | RFM                                                       |
| Growth Potential   | Assigned if not matched above, r_score >= 4, high_engagement is true, and either f_score <= 2 or m_score <= 2. `high_engagement` means sessions_30d >= 8 or campaign_clicks_30d >= 1.      | RFM + app/web activity + campaign engagement              |
| Discount-Sensitive | Assigned if not matched above, avg_discount_pct_180d >= 0.34, campaign_clicks_30d > 0, and r_score >= 2.                                                                                   | Discount usage + campaign engagement + recency            |
| Service Recovery   | Assigned if not matched above, r_score >= 2, and service_friction is true. service_friction means (ticket_count_90d >= 1 and negative_ticket_rate_90d >= 0.50) or return_rate_180d > 0.25. | Support complaints + ticket sentiment + returns + recency |
| Dormant At-Risk    | Assigned if not matched above, r_score <= 2, and sessions_30d <= 2.                                                                                                                        | Recency + app/web activity                                |
| Mixed Watchlist    | Catch-all segment for customers not captured by any higher-priority rule above.                                                                                                            | Residual mixed signal set                                 |

## Segment-Level Evidence

| segment_name       |   customers |   observed_churn_pct |   avg_recency_days |   avg_frequency_180d |   avg_monetary_180d |   avg_sessions_30d |   avg_ticket_count_90d |   avg_return_rate_180d |
|:-------------------|------------:|---------------------:|-------------------:|---------------------:|--------------------:|-------------------:|-----------------------:|-----------------------:|
| Dormant At-Risk    |         427 |                 86.4 |           189.817  |             0.779859 |             597.812 |            1.01171 |             0.00234192 |             0.0163934  |
| Discount-Sensitive |         112 |                 60.7 |            85.3839 |             1.60714  |            1068.51  |            6.47321 |             0.25       |             0.077381   |
| Service Recovery   |         200 |                 56.5 |            69.48   |             1.81     |            1421.51  |            5.28    |             0.88       |             0.31625    |
| Mixed Watchlist    |         851 |                 53.9 |           104.145  |             1.34195  |             991.309 |            5.54877 |             0.039953   |             0.00705053 |
| Growth Potential   |         221 |                 21.7 |            20.0136 |             1.0362   |             685.719 |            8.29412 |             0.266968   |             0.0859729  |
| Loyal Core         |         290 |                 15.2 |            21.4172 |             2.32414  |            1611.75  |            7.15862 |             0.568966   |             0.150718   |
| Champions          |         299 |                  8.7 |            19.8328 |             3.04348  |            2413.32  |            7.55853 |             0.29097    |             0          |

## Expected Business Value

The table below is a directional estimate of where value is sitting in the portfolio. `estimated_value_at_risk_inr` is calculated as:

`customers * avg_monetary_180d * observed_churn_rate`

It is not an uplift estimate, but it does show where churn intersects with recent spend strongly enough to matter financially.

| segment_name       |   estimated_value_at_risk_inr |   priority_index | expected_business_value                                                                                |
|:-------------------|------------------------------:|-----------------:|:-------------------------------------------------------------------------------------------------------|
| Mixed Watchlist    |                        455011 |            437.7 | Broad middle pool with real revenue at risk, but action should stay lightweight until signal sharpens. |
| Dormant At-Risk    |                        220592 |             86.6 | Large at-risk pool; value depends on cheap win-back because current engagement is weak.                |
| Service Recovery   |                        160631 |            168.1 | High save potential because customers still engage, spend materially, and show fixable friction.       |
| Discount-Sensitive |                         72659 |            193.9 | Moderate churn with promotion response; recoverable value if discounting stays disciplined.            |
| Loyal Core         |                         70917 |            166.3 | Healthy repeat buyers worth preserving with low-friction nudges before cadence softens.                |
| Champions          |                         62746 |            359.2 | Low churn but very high spend; protect margin and loyalty rather than overspend on saves.              |
| Growth Potential   |                         32914 |             76.9 | Lower current spend but strong engagement; good upside from category expansion.                        |

## Recommended Actions

| segment_name       | recommended_action                                             |   estimated_cost_per_customer_inr | rationale                                                     |
|:-------------------|:---------------------------------------------------------------|----------------------------------:|:--------------------------------------------------------------|
| Champions          | VIP early-access message with zero-discount content            |                                 5 | Protect margin while rewarding the best customers.            |
| Loyal Core         | Timed replenishment reminder plus free shipping                |                                12 | Keep a healthy repeat cadence without over-discounting.       |
| Growth Potential   | Cross-category starter bundle or personalized routine builder  |                                18 | Broaden basket depth while engagement is still high.          |
| Discount-Sensitive | Minimum-basket bundle discount                                 |                                25 | Speak to price sensitivity without collapsing AOV.            |
| Service Recovery   | Agent callback plus replacement or free-shipping credit        |                                30 | Fix resolvable friction before it becomes irreversible churn. |
| Dormant At-Risk    | Win-back free-shipping reminder with product-specific creative |                                12 | Reactivate customers before inactivity hardens further.       |
| Mixed Watchlist    | Low-cost reminder or content-led nurture                       |                                 8 | Maintain contact while gathering more signal.                 |

## Budgeted Plan

Assumed campaign budget: **₹12,000**

| segment_name       |   customers_in_segment |   estimated_cost_per_customer |   targeted_customers_under_budget |   planned_spend_inr |
|:-------------------|-----------------------:|------------------------------:|----------------------------------:|--------------------:|
| Service Recovery   |                    200 |                            30 |                               200 |                6000 |
| Dormant At-Risk    |                    427 |                            12 |                               427 |                5124 |
| Discount-Sensitive |                    112 |                            25 |                                35 |                 875 |
| Growth Potential   |                    221 |                            18 |                                 0 |                   0 |
| Mixed Watchlist    |                    851 |                             8 |                                 0 |                   0 |
| Loyal Core         |                    290 |                            12 |                                 0 |                   0 |
| Champions          |                    299 |                             5 |                                 0 |                   0 |

With a budget of ₹12,000, the first segment to prioritize is **Service Recovery**. It combines a materially high churn rate (56.5%) with meaningful recent value (₹1422 average 180-day spend) and still-visible engagement (5.3 sessions). Dormant At-Risk customers churn even more heavily (86.4%), but their average engagement is far lower (1.0 sessions), which makes them a second-wave priority rather than the first rupee spent.

## Practical Guardrails

1. Do not spend discount budget on `Champions` first. Their churn is already the lowest, so a margin-light loyalty treatment is enough.
2. Treat `Service Recovery` as an operations-led retention queue, not a coupon queue.
3. For `Discount-Sensitive`, use basket-building offers instead of flat percentage discounts.
4. For `Dormant At-Risk`, suppress repeated offers after one failed win-back touch to avoid wasting spend on fully inactive customers.


# Manual Review Cases

The ten customers below are intentionally selected because the automated segment assignment is not enough on its own. Each case mixes valuable signal with a meaningful contradiction.

| customer_id   | segment_name       |   recency_days |   frequency_180d |   monetary_180d |   sessions_30d |   campaign_clicks_30d |   ticket_count_90d |   return_rate_180d | review_reason                                                                                                      | recommended_decision                                                                          |
|:--------------|:-------------------|---------------:|-----------------:|----------------:|---------------:|----------------------:|-------------------:|-------------------:|:-------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------|
| CUST01148     | Dormant At-Risk    |            128 |                7 |         5448    |              1 |                     0 |                  0 |           0        | High historical spend but now dormant; expensive to ignore yet already cold.                                       | Use a high-touch win-back with capped incentive, then suppress if there is still no response. |
| CUST00841     | Dormant At-Risk    |             99 |                6 |         5236.09 |              2 |                     0 |                  0 |           0.166667 | High historical spend but now dormant; expensive to ignore yet already cold.                                       | Use a high-touch win-back with capped incentive, then suppress if there is still no response. |
| CUST00966     | Growth Potential   |             40 |                1 |         2196.79 |              4 |                     2 |                  1 |           0        | Strong recent engagement but friction exists, so a pure upsell message could backfire.                             | Fix the service issue first, then cross-sell only after the customer is stable.               |
| CUST00071     | Growth Potential   |             32 |                1 |         2062.5  |              1 |                     2 |                  1 |           0        | Strong recent engagement but friction exists, so a pure upsell message could backfire.                             | Fix the service issue first, then cross-sell only after the customer is stable.               |
| CUST01295     | Service Recovery   |            121 |                3 |        11655.3  |             11 |                     0 |                  0 |           0.333333 | Support friction is present, but engagement remains strong enough that the customer may return without a discount. | Give a service-led callback or replacement credit rather than a blanket coupon.               |
| CUST01027     | Service Recovery   |             58 |                4 |         4029.95 |             11 |                     0 |                  1 |           0        | Support friction is present, but engagement remains strong enough that the customer may return without a discount. | Give a service-led callback or replacement credit rather than a blanket coupon.               |
| CUST00133     | Discount-Sensitive |             54 |                6 |         4776.26 |              5 |                     1 |                  0 |           0        | Looks price-sensitive, but there is no sign of operational friction or poor product fit.                           | Use a minimum-basket bundle instead of a deep one-off discount.                               |
| CUST02261     | Discount-Sensitive |             55 |                3 |         2248.93 |              4 |                     1 |                  0 |           0        | Looks price-sensitive, but there is no sign of operational friction or poor product fit.                           | Use a minimum-basket bundle instead of a deep one-off discount.                               |
| CUST01424     | Mixed Watchlist    |             93 |                1 |         2318.97 |              3 |                     0 |                  0 |           0        | Signals conflict: middle-of-pack RFM plus enough activity to justify a human check.                                | Route to CRM analyst review before spending meaningful budget.                                |
| CUST00131     | Mixed Watchlist    |             67 |                1 |         2157.15 |              4 |                     0 |                  0 |           0        | Signals conflict: middle-of-pack RFM plus enough activity to justify a human check.                                | Route to CRM analyst review before spending meaningful budget.                                |


In [ ]:
# Rebuild from the command line with: python build_part2.py